# Segmenting Experiment 2 Free Condition

In [ ]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
from data.spellcheck import spellcheck_recalls_dl
from data.prolific_quality import filter_sessions_by_quality, dedupe_replayed_sessions
from data.response_cleaning import clean_responses

In [ ]:
# Load and clean Experiment 2 data (same pipeline as exp1_exp2_comparison.ipynb)
data_dir = Path('data/data_storage')
data_path = max(data_dir.glob('dirfru_trials*.csv'), key=lambda p: p.stat().st_mtime)
raw = pd.read_csv(data_path, low_memory=False)
raw['prolific_pid'] = raw['prolific_pid'].astype(str).str.strip()
raw = raw[~raw['prolific_pid'].isin(['6760981a3e2500d4057d41be'])].copy()
raw = dedupe_replayed_sessions(raw)

df = raw[raw['type'].isin(['WORD', 'REC_WORD'])].copy()
for col in ['serial_position', 'list', 'session', 'l_length', 'rt']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

word_rows = df[df['type'] == 'WORD'].copy()
word_rows['word_key'] = word_rows['word'].str.lower().str.strip()
word_maps = word_rows.set_index(['prolific_pid', 'session', 'list', 'word_key'])['serial_position'].to_dict()

def recover_sp(row):
    if row['type'] != 'REC_WORD':
        return row['serial_position']
    if pd.isna(row['rec_word']) or str(row['rec_word']).strip() in ['', 'null']:
        return 99
    key = (row['prolific_pid'], row['session'], row['list'], str(row['rec_word']).lower().strip())
    return word_maps.get(key, 88)

df, cleaning_log, removed_responses = clean_responses(df)
df['serial_position'] = df.apply(recover_sp, axis=1)
df = spellcheck_recalls_dl(df)
df = filter_sessions_by_quality(df, raw)
df = df[df['study_id'].isin(['6a94e3286f8fd3491209112a', '6a94e406a70405ad5a421026', '6a94e5555b00e0e80a3ac706',
                             '6a94e598703277b3ace1b216', '6a94e5d410e6c6c4deca61c7'])]

p0 = set(df[df['session'] == 0]['prolific_pid']); p1 = set(df[df['session'] == 1]['prolific_pid']); p2 = set(df[df['session'] == 2]['prolific_pid'])
filtered_df = df[(df['prolific_pid'].isin(p0 & p1 & p2)) & (df['session'] >= 1)].copy()
filtered_df['initiation_condition'] = filtered_df['initiation_condition'].map({'primacy': 'start', 'recency': 'end', 'free': 'free'})
filtered_df.drop_duplicates(['prolific_pid', 'initiation_condition']).groupby('initiation_condition')['prolific_pid'].nunique()

In [ ]:
# Probability of first recall per session, lists initiated with a correct recall only (as pfr_sess in R1G_MFR)
def pfr_sess(data, ll=20):
    pfr = np.zeros(ll)
    rec_evs = data[data['type'] == 'REC_WORD']
    subtract = 0
    for i in data.list.unique():
        sp = rec_evs[rec_evs['list'] == i].serial_position.to_numpy()
        if len(sp) > 1 and sp[0] > 0 and sp[0] <= ll:
            pfr[int(sp[0]) - 1] += 1
        else:
            subtract += 1
    return pfr / (len(data.list.unique()) - subtract)

rows = []
for (pid, sess, cond), data in filtered_df.groupby(['prolific_pid', 'session', 'initiation_condition']):
    rows.append((pid, sess, cond) + tuple(pfr_sess(data)))
pfr_data = pd.DataFrame(rows, columns=['subject', 'session', 'condition'] + [f'sp_{x}' for x in range(1, 21)])
pfr_bsa = pfr_data.groupby(['subject', 'condition'])[[f'sp_{x}' for x in range(1, 21)]].mean().reset_index()

In [ ]:
# Recall initiation groups, same rule as r1_groups_partition (first/last 4 serial positions, 2/3 threshold)
pfr_bsa['prim_pfr'] = pfr_bsa[[f'sp_{x}' for x in range(1, 5)]].sum(axis=1)
pfr_bsa['rec_pfr'] = pfr_bsa[[f'sp_{x}' for x in range(17, 21)]].sum(axis=1)
pfr_bsa['group'] = np.where(pfr_bsa.prim_pfr >= 2/3, 'Primacy', np.where(pfr_bsa.rec_pfr >= 2/3, 'Recency', 'Other'))
pd.crosstab(pfr_bsa.condition, pfr_bsa.group, margins=True)

In [ ]:
# Free condition participants, ordered by proportion of primacy initiations
pfr_bsa[pfr_bsa.condition == 'free'].sort_values('prim_pfr', ascending=False)[['subject', 'prim_pfr', 'rec_pfr', 'group']].round(2)

In [ ]:
# Experiment 1 groups in the 20-1 condition, for comparison
exp1_groups = pd.read_csv('../R1G_MFR/analyses/dataframes/r1_groups.csv')
exp1_groups[exp1_groups.condition == '20-1'].strat.value_counts()

In [ ]:
# Murdock (1962) 20-1 first recalls, lists initiated with a correct recall (no subject IDs, one row per list)
rows = [list(map(int, l.split())) for l in open('data/murdock1962/Murd62.data/Murd62/fr20-1.txt') if l.strip()]
first = np.array([r[0] for r in rows])
murd = first[(first >= 1) & (first <= 20)]
print(f'{len(rows)} lists, {len(murd)} initiated with a correct recall')
print(f'P(SP1-4) = {np.mean(murd <= 4):.3f}, P(SP17-20) = {np.mean(murd >= 17):.3f}')

In [ ]:
# Murdock groups, assuming rows are ordered by subject in blocks of 80 lists
murd_groups = []
for i in range(0, len(first), 80):
    b = first[i:i + 80]
    b = b[(b >= 1) & (b <= 20)]
    prim, rec = np.mean(b <= 4), np.mean(b >= 17)
    murd_groups.append(('Primacy' if prim >= 2/3 else 'Recency' if rec >= 2/3 else 'Other', round(prim, 2), round(rec, 2)))
murd_groups = pd.DataFrame(murd_groups, columns=['group', 'prim_pfr', 'rec_pfr'])
print(murd_groups.group.value_counts())
murd_groups

In [ ]:
# First recall comparison across Murdock, Experiment 2 Free and Experiment 1 (20-1)
free = pfr_bsa[pfr_bsa.condition == 'free']
exp1_201 = exp1_groups[exp1_groups.condition == '20-1']
pd.DataFrame({
    'P(SP1-4)': [np.mean(murd <= 4), free.prim_pfr.mean(), exp1_201.prim_pfr.mean()],
    'P(SP17-20)': [np.mean(murd >= 17), free.rec_pfr.mean(), exp1_201.rec_pfr.mean()],
    'n': [len(murd_groups), len(free), len(exp1_201)],
}, index=['Murdock (1962) 20-1', 'Exp 2 Free', 'Exp 1 20-1']).round(3)